# NB_bishop_ch18_normalizing_flows

**Chapter 18 — Normalizing Flows**

This notebook implements a simple coupling layer (RealNVP-style), trains it on 2D synthetic distributions (moons, circles), visualizes the learned density, and introduces the neural ODE concept.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_18/NB_bishop_ch18_normalizing_flows.ipynb)

In [ ]:
# Install dependencies (Colab-safe)
# !pip install tensorflow numpy matplotlib scikit-learn

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import make_moons, make_circles

print(f'TensorFlow version: {tf.__version__}')

In [ ]:
# Chart styling setup
matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.loc'] = 'upper center'
matplotlib.rcParams['legend.framealpha'] = 0.0

def save_fig(fig, name, chart_dir='../../charts'):
    """Save figure with transparent background, no grid, legend outside bottom."""
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            ncol = min(len(legend.get_texts()), 4)
            legend._ncols = ncol
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

## 1. Background: Change of Variables

A normalizing flow transforms a simple base distribution $p_0(\mathbf{z})$ through a sequence of invertible mappings $f_1, f_2, \ldots, f_K$ to produce a complex distribution:

$$\log p(\mathbf{x}) = \log p_0(\mathbf{z}_0) - \sum_{k=1}^{K} \log \left| \det \frac{\partial f_k}{\partial \mathbf{z}_{k-1}} \right|$$

In [ ]:
# Generate 2D datasets
np.random.seed(42)
X_moons, _ = make_moons(n_samples=5000, noise=0.05)
X_circles, _ = make_circles(n_samples=5000, noise=0.05, factor=0.5)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], s=3, alpha=0.5, color='#1a3a6e')
axes[0].set_title('Two Moons')
axes[0].set_aspect('equal')
axes[1].scatter(X_circles[:, 0], X_circles[:, 1], s=3, alpha=0.5, color='#cd0000')
axes[1].set_title('Concentric Circles')
axes[1].set_aspect('equal')
fig.suptitle('Target 2D Distributions', fontsize=14)
fig.tight_layout()
plt.show()

## 2. Affine Coupling Layer (RealNVP)

In [ ]:
class AffineCouplingLayer(keras.layers.Layer):
    """RealNVP-style affine coupling layer for 2D data.
    
    Splits input into two halves. One half is passed through unchanged,
    the other is affinely transformed using s and t networks conditioned
    on the unchanged half.
    """
    def __init__(self, mask, hidden_units=64, **kwargs):
        super().__init__(**kwargs)
        self.mask = tf.constant(mask, dtype=tf.float32)
        
        # Scale network
        self.s_net = keras.Sequential([
            layers.Dense(hidden_units, activation='relu'),
            layers.Dense(hidden_units, activation='relu'),
            layers.Dense(2, activation='tanh')  # bounded scale
        ])
        # Translation network
        self.t_net = keras.Sequential([
            layers.Dense(hidden_units, activation='relu'),
            layers.Dense(hidden_units, activation='relu'),
            layers.Dense(2)
        ])
    
    def call(self, x, reverse=False):
        x_masked = x * self.mask
        s = self.s_net(x_masked) * (1 - self.mask)
        t = self.t_net(x_masked) * (1 - self.mask)
        
        if not reverse:
            # Forward: z = x * exp(s) + t
            z = x * tf.exp(s) + t * (1 - self.mask)
            z = z * (1 - self.mask) + x_masked
            log_det = tf.reduce_sum(s, axis=-1)
            return z, log_det
        else:
            # Inverse: x = (z - t) * exp(-s)
            x_out = (x - t * (1 - self.mask)) * tf.exp(-s)
            x_out = x_out * (1 - self.mask) + x_masked
            return x_out

print('AffineCouplingLayer defined.')

In [ ]:
class RealNVP(keras.Model):
    """RealNVP normalizing flow model for 2D density estimation."""
    def __init__(self, n_coupling_layers=6, hidden_units=64, **kwargs):
        super().__init__(**kwargs)
        self.coupling_layers = []
        masks = [[1.0, 0.0], [0.0, 1.0]]  # alternating masks
        for i in range(n_coupling_layers):
            mask = masks[i % 2]
            self.coupling_layers.append(
                AffineCouplingLayer(mask, hidden_units=hidden_units)
            )
    
    def call(self, x):
        """Forward pass: data -> latent, returns z and log_det_jacobian."""
        log_det_sum = 0.0
        z = x
        for layer in self.coupling_layers:
            z, log_det = layer(z, reverse=False)
            log_det_sum += log_det
        return z, log_det_sum
    
    def inverse(self, z):
        """Inverse pass: latent -> data."""
        x = z
        for layer in reversed(self.coupling_layers):
            x = layer(x, reverse=True)
        return x
    
    def log_prob(self, x):
        """Compute log probability of data under the model."""
        z, log_det = self(x)
        # Base distribution: standard normal
        log_pz = -0.5 * tf.reduce_sum(z**2, axis=-1) - np.log(2 * np.pi)
        return log_pz + log_det

print('RealNVP model defined.')

## 3. Training on Two Moons

In [ ]:
# Create and train model on moons dataset
tf.random.set_seed(42)
flow_model = RealNVP(n_coupling_layers=8, hidden_units=128)

optimizer = keras.optimizers.Adam(learning_rate=1e-3)

# Prepare data
X_data = X_moons.astype(np.float32)
dataset = tf.data.Dataset.from_tensor_slices(X_data).shuffle(5000).batch(256)

losses = []
EPOCHS = 200

for epoch in range(EPOCHS):
    epoch_losses = []
    for batch in dataset:
        with tf.GradientTape() as tape:
            log_prob = flow_model.log_prob(batch)
            loss = -tf.reduce_mean(log_prob)  # negative log-likelihood
        grads = tape.gradient(loss, flow_model.trainable_variables)
        optimizer.apply_gradients(zip(grads, flow_model.trainable_variables))
        epoch_losses.append(loss.numpy())
    losses.append(np.mean(epoch_losses))
    
    if (epoch + 1) % 50 == 0:
        print(f'Epoch {epoch+1}/{EPOCHS} — NLL: {losses[-1]:.4f}')

In [ ]:
# Plot training loss
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(losses, color='#1a3a6e', linewidth=1.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Negative Log-Likelihood')
ax.set_title('Normalizing Flow Training (Two Moons)')
fig.tight_layout()
save_fig(fig, 'bishop_ch18_flow_training')
plt.show()

## 4. Visualize Learned Density

In [ ]:
# Evaluate log-density on a grid
xmin, xmax = X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5
ymin, ymax = X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5

xx, yy = np.meshgrid(np.linspace(xmin, xmax, 200), np.linspace(ymin, ymax, 200))
grid_points = np.column_stack([xx.ravel(), yy.ravel()]).astype(np.float32)

# Evaluate in batches
log_probs = []
for i in range(0, len(grid_points), 1000):
    batch = grid_points[i:i+1000]
    lp = flow_model.log_prob(batch).numpy()
    log_probs.append(lp)
log_probs = np.concatenate(log_probs)
density = np.exp(log_probs).reshape(xx.shape)

print(f'Density range: [{density.min():.4f}, {density.max():.4f}]')

In [ ]:
# Plot learned density and samples
# Generate samples from the flow
z_samples = tf.random.normal([3000, 2])
x_samples = flow_model.inverse(z_samples).numpy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Original data
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], s=3, alpha=0.5, color='#1a3a6e')
axes[0].set_title('Training Data')
axes[0].set_aspect('equal')

# Learned density contour
axes[1].contourf(xx, yy, density, levels=50, cmap='Blues')
axes[1].set_title('Learned Density')
axes[1].set_aspect('equal')

# Generated samples
axes[2].scatter(x_samples[:, 0], x_samples[:, 1], s=3, alpha=0.5, color='#cd0000')
axes[2].set_title('Generated Samples')
axes[2].set_aspect('equal')

fig.suptitle('Normalizing Flow — Two Moons', fontsize=14)
fig.tight_layout()
save_fig(fig, 'bishop_ch18_learned_density')
plt.show()

## 5. Flow Transformation Visualization

In [ ]:
# Visualize how the flow transforms the base distribution step by step
z_vis = tf.random.normal([2000, 2])

# Apply coupling layers one by one
intermediates = [z_vis.numpy()]
z_current = z_vis
for layer in flow_model.coupling_layers:
    z_current = layer(z_current, reverse=True)
    intermediates.append(z_current.numpy())

n_steps = len(intermediates)
fig, axes = plt.subplots(1, min(n_steps, 5), figsize=(16, 3.5))
step_indices = np.linspace(0, n_steps - 1, min(n_steps, 5), dtype=int)

for i, idx in enumerate(step_indices):
    data = intermediates[idx]
    axes[i].scatter(data[:, 0], data[:, 1], s=2, alpha=0.4, color='#1a3a6e')
    title = 'Base (N(0,I))' if idx == 0 else f'After Layer {idx}'
    if idx == n_steps - 1:
        title = 'Final (Target)'
    axes[i].set_title(title)
    axes[i].set_aspect('equal')

fig.suptitle('Flow Transformation: Base to Target', fontsize=14)
fig.tight_layout()
plt.show()

## 6. Training on Concentric Circles

In [ ]:
# Train a second flow on circles
tf.random.set_seed(123)
flow_circles = RealNVP(n_coupling_layers=8, hidden_units=128)
opt_circles = keras.optimizers.Adam(learning_rate=1e-3)

X_circ = X_circles.astype(np.float32)
ds_circ = tf.data.Dataset.from_tensor_slices(X_circ).shuffle(5000).batch(256)

losses_circ = []
for epoch in range(200):
    el = []
    for batch in ds_circ:
        with tf.GradientTape() as tape:
            lp = flow_circles.log_prob(batch)
            loss = -tf.reduce_mean(lp)
        grads = tape.gradient(loss, flow_circles.trainable_variables)
        opt_circles.apply_gradients(zip(grads, flow_circles.trainable_variables))
        el.append(loss.numpy())
    losses_circ.append(np.mean(el))
    if (epoch + 1) % 50 == 0:
        print(f'Circles — Epoch {epoch+1}/200 — NLL: {losses_circ[-1]:.4f}')

In [ ]:
# Visualize circles density
xx2, yy2 = np.meshgrid(np.linspace(-1.5, 1.5, 200), np.linspace(-1.5, 1.5, 200))
grid2 = np.column_stack([xx2.ravel(), yy2.ravel()]).astype(np.float32)

lp2 = []
for i in range(0, len(grid2), 1000):
    lp2.append(flow_circles.log_prob(grid2[i:i+1000]).numpy())
density2 = np.exp(np.concatenate(lp2)).reshape(xx2.shape)

z_circ = tf.random.normal([3000, 2])
x_circ_gen = flow_circles.inverse(z_circ).numpy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].scatter(X_circles[:, 0], X_circles[:, 1], s=3, alpha=0.5, color='#1a3a6e')
axes[0].set_title('Training Data'); axes[0].set_aspect('equal')
axes[1].contourf(xx2, yy2, density2, levels=50, cmap='Reds')
axes[1].set_title('Learned Density'); axes[1].set_aspect('equal')
axes[2].scatter(x_circ_gen[:, 0], x_circ_gen[:, 1], s=3, alpha=0.5, color='#cd0000')
axes[2].set_title('Generated Samples'); axes[2].set_aspect('equal')
fig.suptitle('Normalizing Flow — Concentric Circles', fontsize=14)
fig.tight_layout()
plt.show()

## 7. Neural ODE Concept Demo

Neural ODEs define continuous-depth transformations: $\frac{d\mathbf{z}(t)}{dt} = f(\mathbf{z}(t), t; \theta)$.

This is a conceptual demo using simple Euler integration.

In [ ]:
# Simple Neural ODE concept: Euler integration of a learned vector field
class SimpleNeuralODE(keras.Model):
    """Toy neural ODE for 2D flow using Euler integration."""
    def __init__(self, hidden=64, n_steps=20, **kwargs):
        super().__init__(**kwargs)
        self.n_steps = n_steps
        self.dt = 1.0 / n_steps
        # Time-independent velocity field f(z)
        self.velocity = keras.Sequential([
            layers.Dense(hidden, activation='tanh'),
            layers.Dense(hidden, activation='tanh'),
            layers.Dense(2)
        ])
    
    def forward(self, z0):
        """Integrate forward: z0 -> z1 (base -> target direction)."""
        z = z0
        log_det = tf.zeros(tf.shape(z0)[0])
        for _ in range(self.n_steps):
            with tf.GradientTape() as tape:
                tape.watch(z)
                v = self.velocity(z)
            # Approximate log det via trace of Jacobian
            jac = tape.batch_jacobian(v, z)  # (batch, 2, 2)
            trace = tf.linalg.trace(jac)
            z = z + self.dt * v
            log_det += self.dt * trace
        return z, log_det

print('SimpleNeuralODE defined (conceptual demo).')

In [ ]:
# Visualize a learned velocity field (from the RealNVP flow as proxy)
# Show the vector field that transforms base to target
grid_sparse = np.column_stack([
    np.meshgrid(np.linspace(-3, 3, 20), np.linspace(-3, 3, 20))[0].ravel(),
    np.meshgrid(np.linspace(-3, 3, 20), np.linspace(-3, 3, 20))[1].ravel()
]).astype(np.float32)

# Compute displacement vectors using the trained flow inverse
z_grid = tf.constant(grid_sparse)
x_grid = flow_model.inverse(z_grid).numpy()
displacements = x_grid - grid_sparse

fig, ax = plt.subplots(figsize=(8, 8))
ax.quiver(grid_sparse[:, 0], grid_sparse[:, 1],
          displacements[:, 0], displacements[:, 1],
          color='#1a3a6e', alpha=0.7, scale=20)
ax.scatter(X_moons[:500, 0], X_moons[:500, 1], s=5, alpha=0.3, color='#cd0000', label='Target')
ax.set_title('Flow Vector Field (Base to Target)')
ax.set_aspect('equal')
ax.legend()
fig.tight_layout()
plt.show()

## Summary

In this notebook we covered:
- **Change of variables formula** — the theoretical foundation of normalizing flows
- **Affine coupling layers** (RealNVP) — invertible transformations with tractable Jacobians
- **Density estimation** on 2D synthetic data (moons and circles)
- **Step-by-step transformation** from base to target distribution
- **Neural ODE concept** — continuous-time flows via learned vector fields

In [ ]:
# Key takeaways
takeaways = [
    '1. Normalizing flows use invertible transformations with tractable Jacobians.',
    '2. The change of variables formula enables exact log-likelihood computation.',
    '3. Affine coupling layers (RealNVP) are simple yet effective flow building blocks.',
    '4. Flows can model multimodal and non-Gaussian distributions.',
    '5. Neural ODEs generalize flows to continuous-depth transformations.'
]
print('Key Takeaways:')
print('-' * 60)
for t in takeaways:
    print(t)